# ShortGPT - Google Colab Setup (GPU Optimized)

This notebook sets up ShortGPT with GPU acceleration for faster rendering and processing.

In [11]:
# 🎮 Configure GPU for Maximum Performance
import torch
import os

print("🔍 GPU Detection and Configuration...")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 GPU Memory: {gpu_memory:.1f} GB")

    # Configure for optimal GPU usage
    torch.cuda.set_device(0)

    # Auto-detect GPU architecture
    if 'A100' in gpu_name:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'
        torch.cuda.set_per_process_memory_fraction(0.9)
        print("🚀 A100 detected - Maximum performance mode!")
        print("⚡ Expected: 10-15x faster than CPU, 3-4x faster than T4")
    elif 'V100' in gpu_name:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.0'
        torch.cuda.set_per_process_memory_fraction(0.8)
        print("🔥 V100 detected - High performance mode!")
    elif 'T4' in gpu_name:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
        torch.cuda.set_per_process_memory_fraction(0.7)
        print("⚡ T4 detected - Efficient processing mode!")

    # Set CUDA environment variables
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

    print("✅ GPU configured for AI processing")
else:
    print("⚠️  No GPU detected - using CPU mode")
    print("💡 Go to Runtime → Change runtime type → GPU (A100) for best performance")

🔍 GPU Detection and Configuration...
CUDA available: True
🎮 GPU: NVIDIA A100-SXM4-40GB
💾 GPU Memory: 39.6 GB
🚀 A100 detected - Maximum performance mode!
⚡ Expected: 10-15x faster than CPU, 3-4x faster than T4
✅ GPU configured for AI processing


In [ ]:
# 🎬 Install and Configure GPU Video Rendering
!sudo apt-get update -qq
!sudo apt-get install ffmpeg -y -qq

# Fix NumPy/Numba compatibility issue immediately
print("🔧 Fixing NumPy compatibility for Whisper/Numba...")
!pip install numpy==1.26.4 numba==0.59.1 --force-reinstall -q

# Check NVENC support in current FFmpeg
print("🔍 Checking FFmpeg NVENC support...")
try:
    import subprocess
    result = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True, timeout=10)
    nvenc_supported = 'h264_nvenc' in result.stdout
    
    if nvenc_supported:
        print("✅ FFmpeg has NVENC support built-in")
    else:
        print("⚠️  FFmpeg lacks NVENC support")
        print("🔧 Installing FFmpeg with NVENC...")
        
        # Install FFmpeg with NVENC support
        !sudo apt-get install -y software-properties-common
        !sudo add-apt-repository ppa:graphics-drivers/ppa -y
        !sudo apt-get update -qq
        !sudo apt-get install -y nvidia-utils-535 nvidia-driver-535
        
        # Install FFmpeg with CUDA/NVENC
        !wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz
        !tar -xf ffmpeg-release-amd64-static.tar.xz
        !sudo cp ffmpeg-*-amd64-static/ffmpeg /usr/local/bin/ffmpeg
        !sudo cp ffmpeg-*-amd64-static/ffprobe /usr/local/bin/ffprobe
        !rm -rf ffmpeg-*
        
        # Verify NVENC support
        result = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True, timeout=10)
        if 'h264_nvenc' in result.stdout:
            print("✅ FFmpeg with NVENC installed successfully")
        else:
            print("⚠️  NVENC still not available - will use CPU fallback")
            
except Exception as e:
    print(f"⚠️  Error checking FFmpeg: {e}")

# Install GPU-optimized packages
if torch.cuda.is_available():
    print("🔥 Installing GPU-optimized PyTorch and video packages...")
    !pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
    
    # Configure environment for GPU video rendering
    gpu_video_config = {
        'MOVIEPY_GPU': '1',
        'FFMPEG_GPU': '1',
        'MOVIEPY_FFMPEG_GPU': '1',
        'MOVIEPY_CODEC': 'h264_nvenc',     # NVIDIA GPU encoder
        'FFMPEG_CODEC': 'h264_nvenc',      # GPU encoder
        'FFMPEG_PRESET': 'fast',           # GPU-optimized preset
        'FFMPEG_CRF': '23',                # Quality setting
        'IMAGEIO_FFMPEG_GPU': '1',         # Force GPU for imageio
        'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:512',
        'CUDA_LAUNCH_BLOCKING': '0'        # Async GPU operations
    }
    
    for key, value in gpu_video_config.items():
        os.environ[key] = value
    
    print("🎬 Testing GPU video encoding capabilities...")
    !ffmpeg -hide_banner -encoders 2>/dev/null | grep -E '(nvenc|cuda)' | head -5
    
    print("\n✅ GPU video rendering configured!")
    print("🎮 Video encoding: h264_nvenc (GPU) with CPU fallback")
    print("⚡ Expected rendering speed: 5-10x faster than CPU when working")
    print("🔢 NumPy: Fixed to 1.26.4 for Whisper compatibility")
    print("🛡️  Automatic fallback to CPU if GPU encoding fails")
    
else:
    print("📦 Installing CPU-optimized packages...")
    os.environ['MOVIEPY_GPU'] = '0'
    os.environ['FFMPEG_GPU'] = '0'

print("\n🏁 Video rendering setup complete!")

In [13]:
# 🔗 Mount Google Drive and Setup Repository
from google.colab import drive
import os

print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

# Clone or update repository
if not os.path.exists('/content/ShortGPT'):
    print("📂 Cloning ShortGPT repository...")
    !git clone https://github.com/DanielBOnThursday/ShortGPT.git
    %cd /content/ShortGPT/
else:
    %cd /content/ShortGPT/
    print("🔄 Updating repository...")
    !git pull

print("✅ Repository ready!")

🔗 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/ShortGPT
🔄 Updating repository...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.03 KiB | 2.03 MiB/s, done.
From https://github.com/DanielBOnThursday/ShortGPT
   dce021f..06bccf4  stable     -> origin/stable
Updating dce021f..06bccf4
Fast-forward
 gui/ui_tab_short_automation.py | 86 +++++++++++++++++++++++++++++++++++++++++-
 kill_port.sh                   | 12 ++++++
 2 files changed, 96 insertions(+), 2 deletions(-)
 create mode 100755 kill_port.sh
✅ Repository ready!


In [ ]:
# 🛠️ Install Dependencies with GPU Optimizations
print("📦 Installing compatible dependencies...")

# Fix NumPy version conflict for Numba/Whisper compatibility
!pip install numpy==1.26.4 -q --force-reinstall

# Install requirements with compatible versions
!pip install -r requirements.txt -q

# Ensure yt-dlp is properly installed for YouTube video processing
print("📺 Installing yt-dlp for YouTube video support...")
!pip install yt-dlp --upgrade -q

# Test yt-dlp installation
try:
    import yt_dlp
    print("✅ yt-dlp installed and importable")
except ImportError as e:
    print(f"❌ yt-dlp import failed: {e}")
    # Try alternative installation
    !pip install --force-reinstall yt-dlp -q

# Install additional GPU-accelerated packages with fixed versions
if torch.cuda.is_available():
    print("⚡ Installing GPU-accelerated packages...")
    
    # Install compatible versions to avoid conflicts
    !pip install faster-whisper==1.0.2 -q  # Compatible with NumPy 1.26.4
    !pip install numba==0.59.1 -q --force-reinstall  # Compatible with NumPy 1.26.4
    
    # Install GPU-optimized video processing
    !pip install opencv-python-headless==4.8.1.78 -q --force-reinstall
    
    print("🎤 Faster Whisper (GPU): Installed with compatible NumPy")
    print("🎬 OpenCV (GPU ready): Installed with compatible version")
    print("🔢 NumPy: Fixed to 1.26.4 for Numba compatibility")

print("✅ All dependencies installed with compatible versions!")

In [15]:
# 🔑 Load Environment from Google Drive
import os
from dotenv import load_dotenv

env_file_path = '/content/drive/MyDrive/env/.env'

print("🔑 Loading environment variables...")

if os.path.exists(env_file_path):
    try:
        load_dotenv(env_file_path, override=True)
        print(f"✅ Environment loaded from: {env_file_path}")

        # Check API configurations
        apis = {
            '🤖 OpenAI': 'OPENAI_API_KEY',
            '🎤 ElevenLabs': 'ELEVENLABS_API_KEY',
            '🖼️  Pexels': 'PEXELS_API_KEY',
            '☁️  AWS S3': 'AWS_ACCESS_KEY_ID',
            '🔍 SerpAPI': 'SERPAPI_API_KEY'
        }

        print("\n🔑 API Configuration Status:")
        for name, key in apis.items():
            status = "✅ Configured" if os.getenv(key) else "❌ Missing"
            print(f"   {name}: {status}")

    except Exception as e:
        print(f"⚠️  Error loading .env file: {e}")
        print("🔧 Check your .env file format in Google Drive")
else:
    print(f"❌ .env file not found at: {env_file_path}")
    print("📝 Please create your .env file in Google Drive at: /content/drive/MyDrive/env/.env")
    print("\n📄 Example .env file content:")
    print("OPENAI_API_KEY=your-openai-key")
    print("ELEVENLABS_API_KEY=your-elevenlabs-key")
    print("AWS_ACCESS_KEY_ID=your-aws-key")
    print("AWS_SECRET_ACCESS_KEY=your-aws-secret")
    print("AWS_S3_BUCKET=your-bucket-name")

print("\n🚀 Environment setup complete!")

🔑 Loading environment variables...
✅ Environment loaded from: /content/drive/MyDrive/env/.env

🔑 API Configuration Status:
   🤖 OpenAI: ✅ Configured
   🎤 ElevenLabs: ✅ Configured
   🖼️  Pexels: ✅ Configured
   ☁️  AWS S3: ✅ Configured
   🔍 SerpAPI: ✅ Configured

🚀 Environment setup complete!


In [16]:
# 🗃️ Configure AWS S3 (Optional)
import boto3
from botocore.exceptions import ClientError

if os.getenv('AWS_ACCESS_KEY_ID'):
    print("☁️  Configuring AWS S3 integration...")

    try:
        s3_client = boto3.client(
            's3',
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
            region_name=os.getenv('AWS_REGION', 'us-east-2')
        )

        bucket = os.getenv('AWS_S3_BUCKET')
        s3_client.head_bucket(Bucket=bucket)

        print(f"✅ S3 bucket '{bucket}' accessible!")
        print(f"🔗 Videos will be saved to: https://{bucket}.s3.{os.getenv('AWS_REGION', 'us-east-2')}.amazonaws.com/videos/")

        # Set S3 configuration
        os.environ['VIDEO_OUTPUT_S3'] = 'true'

    except Exception as e:
        print(f"⚠️  S3 configuration failed: {e}")
        print("📁 Using local storage for this session")
else:
    print("📁 No AWS credentials found - using local storage")

print("🗃️  Storage configuration complete!")

☁️  Configuring AWS S3 integration...
✅ S3 bucket 'content-automation-assets' accessible!
🔗 Videos will be saved to: https://content-automation-assets.s3.us-east-2.amazonaws.com/videos/
🗃️  Storage configuration complete!


In [ ]:
# 🎯 Configure GPU Video Rendering
import os

if torch.cuda.is_available():
    print("🎯 Configuring GPU video rendering...")
    
    # Set environment variables for GPU rendering
    gpu_env = {
        'MOVIEPY_GPU': '1',
        'FFMPEG_GPU': '1',
        'MOVIEPY_FFMPEG_GPU': '1',
        'MOVIEPY_CODEC': 'h264_nvenc',
        'FFMPEG_CODEC': 'h264_nvenc',
        'FFMPEG_PRESET': 'fast',
        'FFMPEG_CRF': '23',
        'IMAGEIO_FFMPEG_GPU': '1',
        'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:512',
        'CUDA_LAUNCH_BLOCKING': '0'
    }
    
    for key, value in gpu_env.items():
        os.environ[key] = value
    
    print("✅ GPU environment variables configured!")
    print("🎬 Video rendering will use GPU acceleration with CPU fallback")
    print("⚡ Expected 5-10x rendering speedup when GPU is used")
    
    # Test NVENC availability
    import subprocess
    try:
        result = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], 
                              capture_output=True, text=True, timeout=10)
        if 'h264_nvenc' in result.stdout:
            print("✅ NVENC encoder available")
        else:
            print("⚠️  NVENC not available - will use CPU fallback")
    except:
        print("⚠️  Could not test NVENC - will try GPU with fallback")
        
else:
    print("💻 GPU not available - using CPU rendering")
    os.environ['MOVIEPY_GPU'] = '0'
    os.environ['FFMPEG_GPU'] = '0'

print("🎯 Video rendering configuration complete!")

In [ ]:
# 🚀 Launch ShortGPT with Full GPU Acceleration
print("🚀 Launching ShortGPT with GPU acceleration...")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"🎮 GPU: {gpu_name}")
    print(f"⚡ AI Processing: GPU accelerated")
    print(f"🎬 Video Rendering: GPU accelerated (h264_nvenc)")
    print(f"🎤 Speech Processing: GPU accelerated (Faster Whisper)")
    print(f"🖼️  Image Processing: GPU accelerated")

    if 'A100' in gpu_name:
        print(f"🚀 A100 MAXIMUM PERFORMANCE MODE ACTIVE!")
        print(f"   📊 Expected performance: 10-15x faster than CPU")
        print(f"   🎬 Video rendering: Near real-time for short videos")
        print(f"   🎤 Whisper transcription: Almost instantaneous")
else:
    print("💻 Running in CPU mode")

print("\n🌐 Starting ShortGPT interface...")
print("📱 The web interface will open in a new tab")
print("🔗 Public URL will be displayed below for sharing")

# Launch with all optimizations
!python runShortGPTColab.py

🚀 Launching ShortGPT with GPU acceleration...
🎮 GPU: NVIDIA A100-SXM4-40GB
⚡ AI Processing: GPU accelerated
🎬 Video Rendering: GPU accelerated (h264_nvenc)
🎤 Speech Processing: GPU accelerated (Faster Whisper)
🖼️  Image Processing: GPU accelerated
🚀 A100 MAXIMUM PERFORMANCE MODE ACTIVE!
   📊 Expected performance: 10-15x faster than CPU
   🎬 Video rendering: Near real-time for short videos
   🎤 Whisper transcription: Almost instantaneous

🌐 Starting ShortGPT interface...
📱 The web interface will open in a new tab
🔗 Public URL will be displayed below for sharing
🚀 Starting ShortGPT for Google Colab...
🎮 Applying GPU optimizations...
🚀 Applying GPU acceleration patches...
✅ GPU environment variables configured
✅ FFmpeg subprocess patched for GPU
⚠️  MoviePy will be configured when available
✅ CoreEditingEngine patched for GPU
✅ EditingEngine patched for GPU

🎯 GPU Optimization Results: 4/4 patches applied
🎮 GPU Ready: NVIDIA A100-SXM4-40GB
⚡ Video rendering will use NVIDIA GPU acceleratio